# Session 10 - Introduction to Biopython and Final Project

Python for Biosciences - From Zero to Scientific Programming. Mahan Davari.

**Question:** How can we connect our Python functions to records in a biological sequence file?

Outcome: read synthetic FASTA, validate its parsed DNA, reuse core calculations and save one small report. The complete DNA Sequence Analyzer is homework, not this notebook. All identifiers and sequences are invented educational data; no experimental claims follow.

Use the exact [live guide](00_SESSION_GUIDE.md). Start from the downloaded student-course folder or sessions/10 in VS Code. Select the course environment, then Restart Kernel and Run All. No earlier notebook is needed. Offline: use the expected outputs and trace each step on paper.


## 00-10 - Course review

A sequence is an ordered string of bases on one represented DNA strand. DNA uses A, C, G and T; general sequence formats can contain more symbols. GC percentage is the fraction of G and C letters times 100. A sequence length here counts letters (bases), not base pairs inferred from an experiment.

Recall: variables hold text/numbers; conditions reject invalid inputs; loops visit bases/records; dictionaries hold counts; functions return values; modules reuse functions; files preserve results; charts describe values.

Predict: for AAGC, length = 4, G+C = 2, GC = 50%. No function follows from these counts.

Workflow: **synthetic FASTA -> parsed records -> validation -> our functions -> rows -> CSV -> descriptive chart**. Validation must finish before writing output.


## 10-30 - Why Biopython? Meet Seq

Biopython is a specialized Python package for biological sequences and related files. Its import name is Bio. A tested library can handle format details while our own functions still perform the calculations we are learning.

`from Bio.Seq import Seq` imports the Seq type from the Bio.Seq module. A Seq object stores sequence text and provides sequence methods. An ordinary string stores general text. Both support length, indexing and slicing; `str(...)` gives ordinary text for our existing functions. We construct an object without teaching advanced class design.


In [ ]:
from Bio.Seq import Seq
sequence = Seq("ATGCAT")
print("Sequence:", sequence)
print("Length (bases):", len(sequence))
print("First base:", sequence[0])
print("First three bases:", sequence[:3])
print("Ordinary string:", str(sequence))


Expected: ATGCAT, 6, A, ATG and ATGCAT. Zero-based indexing starts at 0; the stop in `[:3]` is excluded.

DNA strands are antiparallel: they run in opposite directions. Complement swaps A with T and C with G. Reverse complement also reverses order, so we show the other strand in the same 5-prime to 3-prime direction. This is a text operation, not evidence of a gene. We keep this introductory example to unambiguous uppercase DNA.


In [ ]:
print("Complement:", sequence.complement())
print("Reverse complement:", sequence.reverse_complement())
# A Seq can store symbols outside our project's alphabet.
print("Seq does not validate our project:", Seq("ATN"))


Expected: TACGTA, ATGCAT, ATN. Do not mistake successful construction for validation. **Checkpoint / Exercise 1:** predict length, slice and reverse complement for AAGC; attempt in your own cell. Keep answers separate in solutions.


## 30-50 - Read FASTA with SeqIO

A FASTA header begins with `>`, then an identifier and optional description. Sequence text follows and may wrap over multiple lines. The next header starts another record. General FASTA may contain DNA ambiguity symbols, RNA or proteins; A/C/G/T-only is our project's simplified subset.

`from Bio import SeqIO` imports the sequence-file module. `SeqIO.parse(handle, "fasta")` supplies records one at a time to a for loop. The loop is familiar; we do not need to write a generator. A record has `id` (first header token), `description` (full header text) and `seq` (a Seq object). The library joins wrapped lines. It is preferable to our limited Session 07 parser for broader format support, but it does not enforce our assignment policy.

Problem: inspect our Session 07 synthetic files. Input: sequences.fasta. Algorithm: locate the course, open the file, iterate records, show identifiers/text/length. Output: two readable records.


In [ ]:
from pathlib import Path
from Bio import SeqIO
# Support the documented course-root and session-folder working directories.
course_root = Path.cwd()
if not (course_root / "datasets").is_dir():
    course_root = Path.cwd().parents[1]
input_path = course_root / "datasets" / "session-07" / "sequences.fasta"
print("Input:", input_path)


In [ ]:
with input_path.open(encoding="utf-8") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        print("ID:", record.id)
        print("Description:", record.description)
        print("Sequence:", str(record.seq))
        print("Length (bases):", len(record.seq))


Expected: Toy_A, description `Toy_A synthetic educational record`, text atgcAT, length 6; Toy_B, GCGC, length 4. SeqIO preserves lowercase here. The two sequence lines in Toy_A are joined in order, not separated by a newline.

**Exercise 2:** repeat with homework.fasta. Predict IDs first. Do not use a single-record reader for a multi-record file. Keep iteration inside the open-file block so the handle stays open.


## 50-60 - Validate and interpret responsibly

Our final-project policy is **nonempty A/C/G/T after parsing, trimming end whitespace and converting to uppercase**. N and internal spaces are rejected. Motifs follow the same normalization and validation policy. SeqIO may remove format whitespace, so validating parsed strings is not checking every raw file-format detail. Our Session 06 helpers use this same normalization. Make it visible in the caller too so the policy is clear.

Use `set(text).issubset(set("ACGT"))` to ask whether every distinct character is allowed. Sets/membership were taught in Session 05; `issubset` is a set method for this whole-set comparison. Empty sets are subsets too, so check nonempty separately.

IDs must be nonempty and unique. Do not silently overwrite duplicate identifiers or skip an invalid record. Check all records before opening the output file. General FASTA can be valid while failing our smaller project policy.


In [ ]:
for text in ["ATGC", "atgc", "ATN", "", "AT GC"]:
    normalized = text.strip().upper()
    valid = normalized != "" and set(normalized).issubset(set("ACGT"))
    print(repr(text), valid)


Expected True for ATGC and atgc after normalization. `repr` displays quotes so empty text and spaces are visible. A missing file is a predictable file error, separate from invalid sequence content. The next short example handles that condition without stopping the lesson.


In [ ]:
missing_path = course_root / "datasets" / "session-07" / "missing.fasta"
try:
    with missing_path.open(encoding="utf-8") as handle:
        missing_records = list(SeqIO.parse(handle, "fasta"))
except FileNotFoundError:
    print("Missing input: check the downloaded folder and filename.")


Expected: a readable missing-input message. A successfully parsed file can still have no records or invalid DNA. A motif occurrence is an exact text match, not proof of a regulatory site. GC alone does not show biological function. These data are synthetic, not observations from an experiment. **Exercise 3:** classify the text inputs and explain why duplicate Same must reject the whole batch.


## 60-70 - Final project specification and rubric

Open [08_PROJECT_INTRODUCTION.md](08_PROJECT_INTRODUCTION.md) and the [final project package](../../final-project/README.md). Read the brief and expected outputs before coding.

Fixed rubric: Python fundamentals/structure 35%; files/output 15%; algorithmic/biological correctness 15%; visualization 10%; errors/tests 10%; documentation/reproducibility 10%; interpretation/limits 5%. Total 100%.

The full project needs multiple FASTA records, validation, A/T/G/C counts, lengths, GC, exact overlapping motifs, CSV and at least two suitable charts. Your functions perform core calculations. The following build intentionally ends at a small CSV; the complete assignment stays independent homework.

Read the output-tool explanations in 08_PROJECT_INTRODUCTION.md: exact decimal formatting, semicolon text joins, DictWriter and dictionary copy. The short prediction cell below isolates the two text operations; the project brief explains exact report columns.


In [ ]:
value = 50.0
parts = ["0", "2"]
print(f"{value:.2f}")
print(";".join(parts))


## 70-105 - A small initial workflow

**Problem:** save length and GC for three invented records. **Input:** Session 07 homework.fasta. **Output:** one CSV with record_id,length_bases,gc_percent.

Plan: read all records; reject zero records, repeated/empty IDs and invalid DNA; call our Session 06 count/GC functions; append rows; save after validation succeeds. Reuse the function definitions below, copied from our earlier work. The standalone script imports the local dna_tools module instead. No complicated orchestration is needed.

Build in small steps: plan (5 minutes), read/validate (8), calculation reuse (7), save (7), independent G-count extension and repair (8). Do not add the full motif/chart assignment during these 35 minutes.


In [ ]:
# Reuse our Session 06 algorithm; input validation happens before these calls.
def count_nucleotides(sequence):
    counts = {"A": 0, "T": 0, "G": 0, "C": 0}
    for base in sequence:
        counts[base] = counts[base] + 1
    return counts

def calculate_gc_content(sequence):
    counts = count_nucleotides(sequence)
    return (counts["G"] + counts["C"]) / len(sequence) * 100


These are the counting and GC calculation bodies from Session 06, with explicit normalization/validation in the caller. They require already valid DNA. A returned dictionary is a value we can store and inspect. Before running, hand-check Seed_A=ATGC: A1/T1/G1/C1, length4, GC50%.


In [ ]:
guided_input = course_root / "datasets" / "session-07" / "homework.fasta"
with guided_input.open(encoding="utf-8") as handle:
    records = list(SeqIO.parse(handle, "fasta"))
print("Read records:", len(records))
if len(records) == 0:
    raise ValueError("No FASTA records found.")


`list(...)` stores the parsed records while the file is open. These examples are tiny; this uses memory proportional to input size and is not a large-file workflow. Expected: 3 records. `raise ValueError(...)` reports an input problem; Session 07 introduced try/except. The script catches that exception and prints a helpful message.


In [ ]:
seen_ids = set()
rows = []
for record in records:
    if record.id == "" or record.id in seen_ids:
        raise ValueError("Empty or duplicate record ID: " + record.id)
    seen_ids.add(record.id)
    sequence_text = str(record.seq).strip().upper()
    if sequence_text == "" or not set(sequence_text).issubset(set("ACGT")):
        raise ValueError("Expected nonempty uppercase A/C/G/T DNA: " + record.id)
    counts = count_nucleotides(sequence_text)
    gc_percent = calculate_gc_content(sequence_text)
    print(record.id, "counts:", counts)
    rows.append([record.id, len(sequence_text), round(gc_percent, 2)])
print("Rows:", rows)


Expected rows: Seed_A,4,50.0; Seed_B,4,0.0; Seed_C,6,100.0. Seed_A counts all 1; Seed_B A2/T2/G0/C0; Seed_C A0/T0/G3/C3. Each loop calls functions afresh, so counts do not leak between records. Round only the saved display value, not intermediate arithmetic.

All records have now passed validation. Only now open the report for writing. `Path.parent` is the containing folder. `mkdir(parents=True, exist_ok=True)` creates missing output folders and accepts an existing one. `writerows` saves our list of rows in order. Opening with `w` replaces a previous successful report on a valid rerun; validation failures must happen earlier.

`round(number, 2)` rounds the numeric value to two decimal places; it does not guarantee trailing zeros when printed. The final project uses the exact text formatting taught in its brief.


In [ ]:
import csv
output_path = course_root / "outputs" / "session10_initial" / "initial_summary.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["record_id", "length_bases", "gc_percent"])
    writer.writerows(rows)
print("Saved:", output_path)


Open the CSV and compare the exact header and all three rows. `newline=""` lets csv.writer handle line endings correctly on Windows. A folder permission problem is an OSError; the standalone script catches it. **Independent Exercise 5:** in your own copy, add one g_count column. Predict its values. Do not open the answers until you have tried.


## 105-115 - Demonstrate, debug and review

Run 03_PRACTICE.py from the course root and compare its output. For a short preview, reuse Session 09 plotting on the stored rows; this chart is not the full project's two-chart solution. IDs are categories, GC percentage is numerical and bounded 0-100. A bar chart compares these invented records. No causal or functional inference is supported.


In [ ]:
# Jupyter command: display this chart inside the notebook.
%matplotlib inline
import matplotlib.pyplot as plt
record_ids = []
gc_values = []
for row in rows:
    record_ids.append(row[0])
    gc_values.append(row[2])
figure, axes = plt.subplots(figsize=(7, 4))
axes.bar(record_ids, gc_values, color="#26738e")
axes.set_xlabel("Synthetic record ID")
axes.set_ylabel("GC (%)")
axes.set_title("GC in three synthetic DNA sequences")
axes.set_ylim(0, 100)
figure.tight_layout()
plt.show()
plt.close(figure)


Expected: bars at 50, 0 and 100, labeled Seed_A/B/C. This is a chart of our invented sequences, not a measured population. A 0% value is a valid computed zero, not missing data. Review [common errors](09_COMMON_ERRORS.md), then answer the [short assessment](07_ASSESSMENT.md). Check one path issue or invalid-input explanation; keep the full test suite for homework.

## 115-120 - Submission and continuation

Follow [05_HOMEWORK.md](05_HOMEWORK.md) and the [project package](../../final-project/README.md). Submit through the teacher's chosen channel by the announced deadline. Record one next step and one unresolved blocker. Continue with careful testing and clearer explanations before attempting larger scientific analyses.

**Summary exercise:** explain this workflow in five sentences, predict all rows by hand and identify at least three missing final-project features. Solutions for ordinary exercises are [separate](solutions/06_SOLUTIONS.md). The complete final project reference is not in this notebook.
